## Tugas 1 CRAWLING DATA

Prio Budi Laksono

210411100177

Preprocessing hasil crawling data dari kompas.com

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

# Daftar user agents yang akan dirotasi
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:89.0) Gecko/20100101 Firefox/89.0',
]

# Fungsi untuk mengambil dan mem-parsing HTML dari URL
def get_soup(url):
    headers = {
        'User-Agent': random.choice(USER_AGENTS)
    }
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return BeautifulSoup(response.text, 'html.parser')
        else:
            print(f"Error {response.status_code} saat mengambil {url}")
            return None
    except Exception as e:
        print(f"Error: {e}")
        return None

def get_article_details(detail_url, category_name):
    detail_soup = get_soup(detail_url)
    if detail_soup:
        # Ambil konten dari div dengan class 'read__content'
        content_div = detail_soup.find('div', class_='read__content')
        content = ' '.join([p.text for p in content_div.find_all('p')]) if content_div else 'Tidak ada isi berita'
        
        # Coba ambil tanggal dari div dengan class 'videoKG-date' terlebih dahulu
        date_tag = detail_soup.find('div', class_='videoKG-date')
        if not date_tag:
            # Jika tidak ditemukan, ambil dari div dengan class 'read__time'
            date_tag = detail_soup.find('div', class_='read__time')
        
        date = date_tag.text.strip().split('-')[-1].strip() if date_tag else 'Tidak ada tanggal'
        
        # Ambil judul artikel
        title_tag = detail_soup.find('h1')
        title = title_tag.text.strip() if title_tag else 'Tidak ada judul'
        
        return {
            'judul': title,
            'isi_berita': content,
            'tanggal': date,
            'kategori': category_name,
        }
    return None


# Fungsi untuk mendapatkan artikel dari suatu kategori
def get_articles(category_url, category_name, max_articles):
    articles = []
    page = 1
    while len(articles) < max_articles:
        url = f'{category_url}?page={page}'
        print(f"Mengambil: {url}")
        soup = get_soup(url)
        if soup is None:
            break
        article_list = soup.find_all('h3', class_='article__title')
        if not article_list:
            print(f"Tidak ada artikel ditemukan di halaman {page}.")
            break
        for article in article_list:
            if len(articles) >= max_articles:
                break
            title_tag = article.find('a')
            detail_url = title_tag['href'] if title_tag else None
            if detail_url:
                article_details = get_article_details(detail_url, category_name)
                if article_details:
                    articles.append(article_details)
        
        sleep_time = random.uniform(3, 6)
#         print(f"Menunggu selama {sleep_time:.2f} detik sebelum mengambil halaman berikutnya...")
        time.sleep(sleep_time)
        
        page += 1
    return articles

# URL Kategori Kompas
categories = {
    'Travel': 'https://www.kompas.com/tag/travel',
    'Timnas Indonesia': 'https://www.kompas.com/tag/timnas-indonesia',
}

max_articles=50

# Mengumpulkan semua data
all_articles = []
for category_name, category_url in categories.items():
    print(f"Menambang kategori {category_name}...")
    articles = get_articles(category_url, category_name, max_articles)
    all_articles.extend(articles)

# Simpan ke dalam DataFrame
df = pd.DataFrame(all_articles)

# Simpan ke dalam file CSV tanpa kolom 'url'
df.to_csv('kompas_articles.csv', index=False)

# Tampilkan 10 data pertama dalam bentuk tabel
print(df.head(10))

print("Proses penambangan data selesai, data tersimpan dalam 'kompas_articles.csv' dan 10 data pertama ditampilkan.")


Menambang kategori Travel...
Mengambil: https://www.kompas.com/tag/travel?page=1
Mengambil: https://www.kompas.com/tag/travel?page=2
Mengambil: https://www.kompas.com/tag/travel?page=3
Mengambil: https://www.kompas.com/tag/travel?page=4
Menambang kategori Timnas Indonesia...
Mengambil: https://www.kompas.com/tag/timnas-indonesia?page=1
Mengambil: https://www.kompas.com/tag/timnas-indonesia?page=2
Mengambil: https://www.kompas.com/tag/timnas-indonesia?page=3
Mengambil: https://www.kompas.com/tag/timnas-indonesia?page=4
                                               judul  \
0  Arti Nama Merek Jetour, Sering Dikira Perusaha...   
1  Ada Pameran Wisata Golden Rama Extra 2024, Taw...   
2  Australia Paling Diminati untuk Liburan Tahun ...   
3  30 Kades Jadi Korban Penipuan Umrah, Uang Rp 1...   
4  Cerita Pengunjung GATF 2024 Dapat Tiket Pesawa...   
5  Promo Tiket Pesawat di GATF 2024 Bank Mandiri,...   
6  Ada Shuttle Gratis di Garuda Indonesia Travel ...   
7  Tiket Terusan Garuda Indo

In [3]:
df=pd.read_csv("kompas_articles.csv")
df.head(1000)

,judul,isi_berita,tanggal,kategori
0,"Arti Nama Merek Jetour, Sering Dikira Perusaha...","JAKARTA, KOMPAS.com - Produsen otomotif China,...","06/12/2024, 13:03 WIB",Travel
1,"Ada Pameran Wisata Golden Rama Extra 2024, Taw...",KOMPAS.com - Pameran perjalanan wisata Experie...,"04/12/2024, 13:07 WIB",Travel
2,Australia Paling Diminati untuk Liburan Tahun ...,KOMPAS.com – Australia resmi dinobatkan sebaga...,"03/12/2024, 17:05 WIB",Travel
3,"30 Kades Jadi Korban Penipuan Umrah, Uang Rp 1...","CIREBON, KOMPAS.com – Sebanyak 30 kepala desa ...","01/12/2024, 16:33 WIB",Travel
4,Cerita Pengunjung GATF 2024 Dapat Tiket Pesawa...,"JAKARTA, KOMPAS.com - Isti, pengunjung Garuda ...","01/12/2024, 07:07 WIB",Travel
...,...,...,...,...
95,"Indonesia Vs Laos 3-3, Pratama Arhan: Kami Aka...","KOMPAS.com - Bek Timnas Indonesia, Pratama Arh...","13/12/2024, 00:32 WIB",Timnas Indonesia
96,Jadwal Piala AFF 2024 Vietnam Vs Indonesia,KOMPAS.com - Timnas Indonesia akan menghadapi ...,"13/12/2024, 00:06 WIB",Timnas Indonesia
97,Klasemen Grup B ASEAN Cup: Indonesia Teratas s...,KOMPAS.com - Timnas Indonesia memuncaki klasem...,"12/12/2024, 22:28 WIB",Timnas Indonesia
98,Hasil Indonesia Vs Laos 3-3: Garuda Ambil Pela...,KOMPAS.com - Timnas Indonesia berbagi hasil im...,"12/12/2024, 22:03 WIB",Timnas Indonesia
